In [2]:
import torch
if not torch.cuda.is_available():
  raise SystemError(
      "No GPU Connected"
  )
device = 'cuda'
print(f"Using device: {device}")

Using device: cuda


In [3]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.3 MB/s eta 0:00:00


In [4]:
import torch
import math
import time
import shutil
import psutil
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

model_id = 'gpt2'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

def get_process_memory_mb():
  """ Return how much RAM is the current program utilizing"""
  ## psutil --> Process and System Utilities
  process = psutil.Process(os.getpid()) ## os.getpid() --> Accessing the current process id!
  return process.memory_info().rss/1024 ** 2 ## rss --> Resident Set Size

def get_gpu_memory_mb():
  """Return current GPU memory in (MB) (if CUDA)"""
  if not torch.cuda.is_available():
    return 0.0
  return torch.cuda.memory_allocated() / 1024 ** 2

def describe_memory(label):
  cpu_mem = get_process_memory_mb()
  gpu_mem = get_gpu_memory_mb()
  print(f"[{label}] CPU memory: {cpu_mem:8.2f} MB | GPU memory: {gpu_mem:8.2f} MB")


@torch.no_grad()
def compute_perplexity(model, tokenizer,text: str) -> float:
  model.eval()
  enc = tokenizer(text, return_tensors="pt").to(device)
  outputs = model(**enc, labels=enc["input_ids"])
  return math.exp(outputs.loss.item())

@torch.no_grad()
def timed_generate(model,tokenizer,prompt:str,max_new_tokens: int = 40, num_runs: int = 3):
  model.eval()
  times = []
  last_output = True

  for i in range(num_runs):
    inputs = tokenizer(prompt,return_tensors = "pt").to(device)
    torch.cuda.empty_cache()
    start = time.perf_counter()
    out = model.generate(**inputs,max_new_tokens=max_new_tokens)
    end = time.perf_counter()
    times.append(end-start)
    last_output = tokenizer.decode(out[0], skip_special_tokens = True)
  avg_time = sum(times)/len(times)
  return avg_time,last_output

Device: cuda


### Loading the 16 bit floating point model

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Tokenizer Loaded: {model_id}")

## Clear the GPU Memory
if torch.cuda.is_available():
  torch.cuda.empty_cache()

print("Loading Baseline FP16 Model...")
describe_memory("Before FP16 load")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = torch.float16,
).to(device)

describe_memory("After FP16 load")
print("Model dtype: ", next(model_fp16.parameters()).dtype)
print("Approx Model foot print ( reported by HF ): ",model_fp16.get_memory_footprint()/1024**2, "MB")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[After FP16 load] CPU memory:  1239.29 MB | GPU memory:   243.48 MB
Model dtype:  torch.float16
Approx Model foot print ( reported by HF ):  237.35009765625 MB


### Loading the 8bit Model

In [6]:
# Clear GPU before Loading
if torch.cuda.is_available():
  torch.cuda.empty_cache()

print("Loading 8 bit quantized Model (BitsAndBytes LLM.int8) ...")
bnb_8bit_config = BitsAndBytesConfig(
  load_in_8bit = True,
  llm_int8_threshold = 0.0,
  llm_int8_enable_fp32_cpu_offload = False,
)
model_8bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = "auto",
    quantization_config = bnb_8bit_config, ## Custom config using Bits and Bytes
    torch_dtype = torch.float16,
)
describe_memory("After 8 bit load")
print("8 bit model type:",type(model_8bit))
print("Model dtype: ", next(model_8bit.parameters()).dtype)
try:
  print("Approx 8-bit footprint:", model_8bit.get_memory_footprint)
except Exception as e:
  print("Get_memory_footprint not available",e)

Loading 8 bit quantized Model (BitsAndBytes LLM.int8) ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[After 8 bit load] CPU memory:  1313.55 MB | GPU memory:   401.40 MB
8 bit model type: <class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
Model dtype:  torch.float16
Approx 8-bit footprint: <bound method PreTrainedModel.get_memory_footprint of GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Linear8bitLt(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear8bitLt(in_features=768, out_features=768, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Linear8bitLt(in_features=768, out_features=3

In [7]:
## Clear the GPU Before loading
if torch.cuda.is_available():
  torch.cuda.empty_cache()

print("Loading a 4-bit NF4 Quantized Model....")
describe_memory("Before 4-bit load")

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True, ## Nestes Quantization for compress scales
    bnb_4bit_quant_type = "nf4", ## NormalFloat4, good for the guassian weights
    bnb_4bit_compute_dtype = torch.float16, ## Compute in fp16 for speed
)
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = "auto",
    quantization_config = bnb_8bit_config, ## Custom config using Bits and Bytes
    torch_dtype = torch.float16,
)
describe_memory("After 4-bit load")
try:
  print("Approx 4-bit footprint:", model_4bit.get_memory_footprint()/1024**2, "MB")
except Exception as e:
  print("Get_memory_footprint not available",e)
print("Model dtype: ", next(model_4bit.parameters()).dtype)

Loading a 4-bit NF4 Quantized Model....
[Before 4-bit load] CPU memory:  1313.56 MB | GPU memory:   401.40 MB


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[After 4-bit load] CPU memory:  1313.97 MB | GPU memory:   559.01 MB
Approx 4-bit footprint: 156.35009765625 MB
Model dtype:  torch.float16


In [9]:
sample_text = (
    "Quantization allows us to run the Large Language Models on the smaller Hardware",
    "By Reducing the precision of the weights and activations."
)
sample_text_str = " ".join(sample_text)
print("Sample Text:",sample_text_str)
print()

ppl_fp16 = compute_perplexity(model_fp16,tokenizer,sample_text_str)
print(f"FP16 Perplexity: {ppl_fp16:.3f}")
print()
ppl_8bit = compute_perplexity(model_8bit,tokenizer,sample_text_str)
print(f"8-bit Perplexity: {ppl_8bit:.3f}")
print()
ppl_4bit = compute_perplexity(model_4bit,tokenizer,sample_text_str)
print(f"4-bit Perplexity: {ppl_4bit:.3f}")
print()

Sample Text: Quantization allows us to run the Large Language Models on the smaller Hardware By Reducing the precision of the weights and activations.



[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


FP16 Perplexity: 157.967

8-bit Perplexity: 157.965

4-bit Perplexity: 157.965



In [12]:
prompt = "In the future, efficient AI systems will"

print("Prompt",prompt)
print()
print("\n --- FP16 Baseline Generation ---")
t_fp16,out_fp16 = timed_generate(model_fp16,tokenizer,prompt)
print(f"Average Time (FP16): {t_fp16:.3f} s")
print(out_fp16)
print()
print("\n --- 8-bit Generation ---")
t_8bit,out_8bit = timed_generate(model_8bit,tokenizer,prompt)
print(f"Average Time (8-bit): {t_8bit:.3f} s")
print(out_8bit)
print()
print("\n --- 4-bit Generation ---")
t_4bit,out_4bit = timed_generate(model_4bit,tokenizer,prompt)
print(f"Average Time (4-bit): {t_4bit:.3f} s")
print(out_4bit)

Prompt In the future, efficient AI systems will


 --- FP16 Baseline Generation ---
Average Time (FP16): 1.090 s
In the future, efficient AI systems will be able to do things like search for and find people, and to do things like search for and find people, and to do things like search for and find people, and to do things like search


 --- 8-bit Generation ---
Average Time (8-bit): 1.933 s
In the future, efficient AI systems will be able to do things like search for and find people, and to search for and find people who are in the same place.

The AI system will also be able to do things like search


 --- 4-bit Generation ---
Average Time (4-bit): 1.781 s
In the future, efficient AI systems will be able to do things like search for and find people, and to search for and find people who are in the same place.

The AI system will also be able to do things like search
